# PostgreSQL SQL Interview Practice — Questions & Executable Solutions

This notebook contains all **80 questions and solutions**, reviewed for **PostgreSQL 18** syntax and the `sql_interview` practice schema.

**Database:** `sql_interview`  
**Connection:** Configure `db_config.ini` once (see `README.md`).  
**Schema:** `public`

Each question includes the relevant tables and columns. Each solution is wrapped in Python and executed through `run_sql()`, so the SQL cells work correctly in a normal VS Code Python/Jupyter kernel.


## Notebook setup

Run these setup cells first.

In [ ]:
# Install packages into the active Jupyter kernel.
%pip install -q psycopg2-binary pandas


In [ ]:
import psycopg2
import pandas as pd
from IPython.display import display
# Import the shared database configuration regardless of whether VS Code
# starts the kernel in coding-exercises/ or coding-exercises/sql/.
from pathlib import Path
import sys
_SQL_DIR = next((p for p in (Path.cwd(), Path.cwd() / "sql")
                 if (p / "db_connection.py").is_file()), None)
if _SQL_DIR is None:
    raise FileNotFoundError(
        "Cannot locate sql/db_connection.py. Open coding-exercises/ or its sql/ folder in VS Code."
    )
if str(_SQL_DIR) not in sys.path:
    sys.path.insert(0, str(_SQL_DIR))
from db_connection import connect, connection_settings

conn = connect(autocommit=True)
print(f"Connected to {connection_settings()['dbname']} as {connection_settings()['user']}")


In [ ]:
def run_sql(sql):
    with conn.cursor() as cur:
        cur.execute(sql)

        if cur.description is None:
            print("Query executed successfully.")
            return

        columns = [desc[0] for desc in cur.description]
        rows = cur.fetchall()

    df = pd.DataFrame(rows, columns=columns)
    display(df)
    #print(f"{len(df)} row(s)")

run_sql(
    "SELECT current_database() AS database, "
    "current_user AS user_name, current_schema() AS schema;"
)


## 1. Joins


### Q01 — Customer Order Summary

Return each customer who has placed at least one order, with customer_id, full_name, number_of_orders, and total_order_amount. Use INNER JOIN.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.customer_id, c.first_name || ' ' || c.last_name AS full_name,
       COUNT(o.order_id) AS number_of_orders, SUM(o.total_amount) AS total_order_amount
FROM customers c JOIN orders o ON o.customer_id=c.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name;
"""

run_sql(sql)


### Q02 — Customers Without Orders

Find customers who have never placed an order. Return customer_id, full_name, city, and signup_date. Use LEFT JOIN.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.customer_id, c.first_name || ' ' || c.last_name AS full_name, c.city, c.signup_date
FROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id
WHERE o.order_id IS NULL;
"""

run_sql(sql)


### Q03 — Orders With Customer and Employee

Return order_id, order_date, customer name, employee name, and total_amount for all orders.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT o.order_id,o.order_date,c.first_name||' '||c.last_name customer_name,
e.first_name||' '||e.last_name employee_name,o.total_amount
FROM orders o JOIN customers c ON c.customer_id=o.customer_id
LEFT JOIN employees e ON e.employee_id=o.sales_employee_id;
"""

run_sql(sql)


### Q04 — Employees and Managers

Using a SELF JOIN, return every employee with their manager's name. Keep employees who have no manager.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT e.employee_id,e.first_name||' '||e.last_name employee_name,
m.first_name||' '||m.last_name manager_name
FROM employees e LEFT JOIN employees m ON m.employee_id=e.manager_id;
"""

run_sql(sql)


### Q05 — Products Never Ordered

Find products that have never appeared in order_items. Return product_id, product_name, category, and unit_price.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price

**Solution:**


In [ ]:
sql = """
SELECT p.product_id,p.product_name,p.category,p.unit_price
FROM products p LEFT JOIN order_items oi ON oi.product_id=p.product_id
WHERE oi.order_item_id IS NULL;
"""

run_sql(sql)


### Q06 — Customer Support Tickets

Return each support ticket with the customer name and the employee assigned to it. Keep unassigned tickets.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT t.ticket_id,c.first_name||' '||c.last_name customer_name,
e.first_name||' '||e.last_name assigned_employee,t.status,t.priority
FROM support_tickets t JOIN customers c ON c.customer_id=t.customer_id
LEFT JOIN employees e ON e.employee_id=t.assigned_employee_id;
"""

run_sql(sql)


### Q07 — Order Line Details

Return order_id, product_name, quantity, unit_price at sale, and line_total for every order item.  
**Tables:** \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active

**Solution:**


In [ ]:
sql = """
SELECT oi.order_id,p.product_name,oi.quantity,oi.unit_price,
oi.quantity*oi.unit_price AS line_total
FROM order_items oi JOIN products p ON p.product_id=oi.product_id;
"""

run_sql(sql)


### Q08 — Same-City Customers

Use a SELF JOIN to find unique pairs of different customers who live in the same city. Do not return reversed duplicate pairs.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date

**Solution:**


In [ ]:
sql = """
SELECT c1.customer_id,c1.first_name||' '||c1.last_name customer1,
c2.customer_id,c2.first_name||' '||c2.last_name customer2,c1.city
FROM customers c1 JOIN customers c2 ON c1.city=c2.city AND c1.customer_id<c2.customer_id;
"""

run_sql(sql)


### Q09 — Projects and Team Members

Return each project and all employees assigned to it, including role_on_project and allocation_pct. Keep projects with no assignments.  
**Tables:** \
**`projects:`** project_id, project_name, start_date, end_date, status \
**`project_assignments:`** assignment_id, project_id, employee_id, role_on_project, allocation_pct \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT p.project_id,p.project_name,e.employee_id,e.first_name||' '||e.last_name employee_name,
pa.role_on_project,pa.allocation_pct
FROM projects p LEFT JOIN project_assignments pa ON pa.project_id=p.project_id
LEFT JOIN employees e ON e.employee_id=pa.employee_id;
"""

run_sql(sql)


### Q10 — Orders With Payment Status

Return every order with customer name and payment status. Keep orders that do not yet have a payment record.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`payments:`** payment_id, order_id, payment_date, amount, payment_method, payment_status

**Solution:**


In [ ]:
sql = """
SELECT o.order_id,c.first_name||' '||c.last_name customer_name,p.payment_status
FROM orders o JOIN customers c ON c.customer_id=o.customer_id
LEFT JOIN payments p ON p.order_id=o.order_id;
"""

run_sql(sql)


## 2. Aggregations


### Q11 — Orders by Status

Count the number of orders in each order status.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY order_count DESC;
"""

run_sql(sql)


### Q12 — Revenue by Customer

Calculate total completed-order revenue per customer and show only customers with more than $2,000 in completed revenue.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.customer_id,c.first_name||' '||c.last_name full_name,SUM(o.total_amount) completed_revenue
FROM customers c JOIN orders o ON o.customer_id=c.customer_id
WHERE o.status='Completed' GROUP BY c.customer_id,c.first_name,c.last_name
HAVING SUM(o.total_amount)>2000 ORDER BY completed_revenue DESC;
"""

run_sql(sql)


### Q13 — Department Salary Statistics

For each department, return employee_count, minimum salary, maximum salary, and average salary.  
**Tables:** \
**`departments:`** department_id, department_name, location \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT d.department_name,COUNT(e.employee_id) employee_count,MIN(e.salary) min_salary,
MAX(e.salary) max_salary,ROUND(AVG(e.salary),2) avg_salary
FROM departments d LEFT JOIN employees e ON e.department_id=d.department_id
GROUP BY d.department_id,d.department_name;
"""

run_sql(sql)


### Q14 — Monthly Revenue

Calculate completed-order revenue by calendar month.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT DATE_TRUNC('month',order_date)::date AS month,SUM(total_amount) revenue
FROM orders WHERE status='Completed' GROUP BY 1 ORDER BY 1;
"""

run_sql(sql)


### Q15 — Conditional Order Counts

For each customer, count total orders, completed orders, cancelled orders, and pending orders using conditional aggregation.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.customer_id,COUNT(o.order_id) total_orders,
COUNT(*) FILTER (WHERE o.status='Completed') completed_orders,
COUNT(*) FILTER (WHERE o.status='Cancelled') cancelled_orders,
COUNT(*) FILTER (WHERE o.status='Pending') pending_orders
FROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id GROUP BY c.customer_id;
"""

run_sql(sql)


### Q16 — Product Sales Performance

For each product, calculate units_sold, number_of_order_lines, and sales_revenue from order_items.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price

**Solution:**


In [ ]:
sql = """
SELECT p.product_id,p.product_name,COALESCE(SUM(oi.quantity),0) units_sold,
COUNT(oi.order_item_id) order_lines,COALESCE(SUM(oi.quantity*oi.unit_price),0) sales_revenue
FROM products p LEFT JOIN order_items oi ON oi.product_id=p.product_id GROUP BY p.product_id,p.product_name;
"""

run_sql(sql)


### Q17 — High-Volume Cities

Return cities with at least 3 customers and show customer_count.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date

**Solution:**


In [ ]:
sql = """
SELECT city,COUNT(*) customer_count FROM customers GROUP BY city HAVING COUNT(*)>=3 ORDER BY customer_count DESC;
"""

run_sql(sql)


### Q18 — Ticket Status Summary

For each ticket priority, count total tickets, open tickets, resolved tickets, and closed tickets.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject

**Solution:**


In [ ]:
sql = """
SELECT priority,COUNT(*) total_tickets,
COUNT(*) FILTER (WHERE status='Open') open_tickets,
COUNT(*) FILTER (WHERE status='Resolved') resolved_tickets,
COUNT(*) FILTER (WHERE status='Closed') closed_tickets
FROM support_tickets GROUP BY priority;
"""

run_sql(sql)


### Q19 — Average Order Value by Segment

For each customer segment, calculate order_count, total_revenue, and average_order_value.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.segment,COUNT(o.order_id) order_count,COALESCE(SUM(o.total_amount),0) total_revenue,
ROUND(AVG(o.total_amount),2) average_order_value
FROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id GROUP BY c.segment;
"""

run_sql(sql)


### Q20 — Department Project Allocation

For each department, calculate the number of distinct employees assigned to projects and the average allocation percentage.  
**Tables:** \
**`departments:`** department_id, department_name, location \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title \
**`project_assignments:`** assignment_id, project_id, employee_id, role_on_project, allocation_pct

**Solution:**


In [ ]:
sql = """
SELECT d.department_name,COUNT(DISTINCT pa.employee_id) assigned_employees,
ROUND(AVG(pa.allocation_pct),2) avg_allocation
FROM departments d LEFT JOIN employees e ON e.department_id=d.department_id
LEFT JOIN project_assignments pa ON pa.employee_id=e.employee_id GROUP BY d.department_id,d.department_name;
"""

run_sql(sql)


## 3. Window Functions


### Q21 — Latest Order per Customer

Return the latest order for each customer using ROW_NUMBER().  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT o.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date DESC,order_id DESC) rn FROM orders o)
SELECT * FROM x WHERE rn=1;
"""

run_sql(sql)


### Q22 — Salary Rank by Department

Rank employees by salary within each department using RANK().  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT employee_id,department_id,first_name,last_name,salary,
RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) salary_rank FROM employees;
"""

run_sql(sql)


### Q23 — Dense Salary Rank

Rank employees by salary within each department using DENSE_RANK() and explain from the output where it differs from RANK().  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT employee_id,department_id,first_name,last_name,salary,
DENSE_RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) dense_salary_rank FROM employees;
"""

run_sql(sql)


### Q24 — Previous Customer Order

For every customer order, show the previous order date and previous order amount using LAG().  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT customer_id,order_id,order_date,total_amount,
LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date,order_id) previous_order_date,
LAG(total_amount) OVER(PARTITION BY customer_id ORDER BY order_date,order_id) previous_order_amount FROM orders;
"""

run_sql(sql)


### Q25 — Next Event Time

For each user in events, show each event and the next event timestamp using LEAD().  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page

**Solution:**


In [ ]:
sql = """
SELECT user_id,event_id,event_type,event_ts,
LEAD(event_ts) OVER(PARTITION BY user_id ORDER BY event_ts,event_id) next_event_ts FROM events;
"""

run_sql(sql)


### Q26 — Running Customer Spend

For each customer, show every order and a running total of order_amount ordered by date and order_id.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT customer_id,order_id,order_date,total_amount,
SUM(total_amount) OVER(PARTITION BY customer_id ORDER BY order_date,order_id ROWS UNBOUNDED PRECEDING) running_spend FROM orders;
"""

run_sql(sql)


### Q27 — Department Salary vs Average

For every employee, show salary, department average salary, and the difference between salary and department average.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT employee_id,department_id,salary,
ROUND(AVG(salary) OVER(PARTITION BY department_id),2) department_avg,
ROUND(salary-AVG(salary) OVER(PARTITION BY department_id),2) difference FROM employees;
"""

run_sql(sql)


### Q28 — Order Sequence Number

Number every customer's orders from earliest to latest using ROW_NUMBER().  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT customer_id,order_id,order_date,
ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date,order_id) order_sequence FROM orders;
"""

run_sql(sql)


### Q29 — Running Product Units

For each product, show order dates and cumulative units sold over time.  
**Tables:** \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
WITH d AS (SELECT oi.product_id,o.order_date,SUM(oi.quantity) units
FROM order_items oi JOIN orders o ON o.order_id=oi.order_id GROUP BY oi.product_id,o.order_date)
SELECT *,SUM(units) OVER(PARTITION BY product_id ORDER BY order_date ROWS UNBOUNDED PRECEDING) cumulative_units FROM d;
"""

run_sql(sql)


### Q30 — Customer Order Gap

For each customer order, calculate the number of days since the customer's previous order.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT o.*,LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date,order_id) prev_date FROM orders o)
SELECT customer_id,order_id,order_date,prev_date,order_date-prev_date AS days_since_previous FROM x;
"""

run_sql(sql)


## 4. CTEs & Subqueries


### Q31 — Above-Average Salary

Find employees whose salary is above the company-wide average salary using a subquery.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT * FROM employees WHERE salary>(SELECT AVG(salary) FROM employees) ORDER BY salary DESC;
"""

run_sql(sql)


### Q32 — Above Department Average

Using a CTE, find employees earning more than their own department's average salary.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
WITH a AS (SELECT department_id,AVG(salary) avg_salary FROM employees GROUP BY department_id)
SELECT e.*,a.avg_salary FROM employees e JOIN a USING(department_id) WHERE e.salary>a.avg_salary;
"""

run_sql(sql)


### Q33 — Customers Above Average Spend

Find customers whose total order spend is greater than the average total spend across customers.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date

**Solution:**


In [ ]:
sql = """
WITH s AS (SELECT customer_id,SUM(total_amount) spend FROM orders GROUP BY customer_id)
SELECT c.customer_id,c.first_name,c.last_name,s.spend FROM s JOIN customers c USING(customer_id)
WHERE s.spend>(SELECT AVG(spend) FROM s) ORDER BY s.spend DESC;
"""

run_sql(sql)


### Q34 — Best-Selling Products

Using one or more CTEs, return products whose units sold are above the average units sold per product.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price

**Solution:**


In [ ]:
sql = """
WITH u AS (SELECT p.product_id,p.product_name,COALESCE(SUM(oi.quantity),0) units
FROM products p LEFT JOIN order_items oi USING(product_id) GROUP BY p.product_id,p.product_name)
SELECT * FROM u WHERE units>(SELECT AVG(units) FROM u) ORDER BY units DESC;
"""

run_sql(sql)


### Q35 — Orders Above Customer Average

Return orders whose total_amount is greater than that customer's own average order amount.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT o.* FROM orders o WHERE total_amount>(SELECT AVG(o2.total_amount) FROM orders o2 WHERE o2.customer_id=o.customer_id);
"""

run_sql(sql)


### Q36 — Departments With High Earners

Return departments containing at least one employee earning more than $110,000. Use EXISTS.  
**Tables:** \
**`departments:`** department_id, department_name, location \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT d.* FROM departments d WHERE EXISTS
(SELECT 1 FROM employees e WHERE e.department_id=d.department_id AND e.salary>110000);
"""

run_sql(sql)


### Q37 — Customers With No Support Tickets

Find customers for whom no support ticket exists. Use NOT EXISTS.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject

**Solution:**


In [ ]:
sql = """
SELECT c.* FROM customers c WHERE NOT EXISTS (SELECT 1 FROM support_tickets t WHERE t.customer_id=c.customer_id);
"""

run_sql(sql)


### Q38 — Project Staffing Pipeline

Use multiple CTEs to calculate headcount and total allocation per project, then return projects with at least 3 assigned employees.  
**Tables:** \
**`projects:`** project_id, project_name, start_date, end_date, status \
**`project_assignments:`** assignment_id, project_id, employee_id, role_on_project, allocation_pct

**Solution:**


In [ ]:
sql = """
WITH s AS (SELECT project_id,COUNT(DISTINCT employee_id) headcount,SUM(allocation_pct) total_allocation
FROM project_assignments GROUP BY project_id)
SELECT p.project_id,p.project_name,s.headcount,s.total_allocation FROM projects p JOIN s USING(project_id) WHERE s.headcount>=3;
"""

run_sql(sql)


### Q39 — Revenue Share by Category

Use CTEs to calculate sales revenue by product category and each category's percentage of total revenue.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price

**Solution:**


In [ ]:
sql = """
WITH c AS (SELECT p.category,SUM(oi.quantity*oi.unit_price) revenue FROM order_items oi JOIN products p USING(product_id) GROUP BY p.category),
t AS (SELECT SUM(revenue) total FROM c)
SELECT c.*,ROUND(100.0*c.revenue/NULLIF(t.total,0),2) revenue_pct FROM c CROSS JOIN t ORDER BY revenue DESC;
"""

run_sql(sql)


### Q40 — Repeat Customers

Use a CTE to identify customers with at least 3 orders and return their first_order_date, latest_order_date, and lifetime_order_value.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT customer_id,COUNT(*) order_count,MIN(order_date) first_order_date,MAX(order_date) latest_order_date,SUM(total_amount) lifetime_order_value
FROM orders GROUP BY customer_id HAVING COUNT(*)>=3)
SELECT c.customer_id,c.first_name,c.last_name,x.* FROM x JOIN customers c USING(customer_id);
"""

run_sql(sql)


## 5. Dates & Time


### Q41 — Orders in Last 30 Days of Dataset

Find orders placed in the 30 days ending on the maximum order_date in the table.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT * FROM orders WHERE order_date BETWEEN (SELECT MAX(order_date)-29 FROM orders) AND (SELECT MAX(order_date) FROM orders) ORDER BY order_date;
"""

run_sql(sql)


### Q42 — Monthly Order Trend

Return month, order_count, and total_revenue for each month.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT DATE_TRUNC('month',order_date)::date AS month,COUNT(*) order_count,SUM(total_amount) total_revenue FROM orders GROUP BY 1 ORDER BY 1;
"""

run_sql(sql)


### Q43 — Month-over-Month Revenue Change

Calculate monthly revenue, previous_month_revenue, absolute_change, and percentage_change.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
WITH m AS (SELECT DATE_TRUNC('month',order_date)::date AS month,SUM(total_amount) revenue FROM orders GROUP BY 1),
x AS (SELECT *,LAG(revenue) OVER(ORDER BY month) previous_month_revenue FROM m)
SELECT *,revenue-previous_month_revenue absolute_change,
ROUND(100.0*(revenue-previous_month_revenue)/NULLIF(previous_month_revenue,0),2) percentage_change FROM x ORDER BY month;
"""

run_sql(sql)


### Q44 — Latest Event per User

Return the latest event for every user.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT e.*,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY event_ts DESC,event_id DESC) rn FROM events e)
SELECT * FROM x WHERE rn=1;
"""

run_sql(sql)


### Q45 — Daily Active Users

Count distinct active users per event_date.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page

**Solution:**


In [ ]:
sql = """
SELECT event_ts::date AS event_date,COUNT(DISTINCT user_id) daily_active_users FROM events GROUP BY 1 ORDER BY 1;
"""

run_sql(sql)


### Q46 — Consecutive Login Days

Find user/date pairs where a login occurred on the immediately preceding calendar day for the same user.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page

**Solution:**


In [ ]:
sql = """
WITH d AS (SELECT DISTINCT user_id,event_ts::date AS login_date FROM events WHERE event_type='login')
SELECT d1.user_id,d1.login_date FROM d d1 JOIN d d2 ON d2.user_id=d1.user_id AND d2.login_date=d1.login_date-1 ORDER BY 1,2;
"""

run_sql(sql)


### Q47 — Days to Resolve Tickets

For resolved or closed tickets, calculate the number of days between created_at and resolved_at.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject

**Solution:**


In [ ]:
sql = """
SELECT ticket_id,created_at,resolved_at,EXTRACT(EPOCH FROM(resolved_at-created_at))/86400.0 days_to_resolve
FROM support_tickets WHERE status IN('Resolved','Closed') AND resolved_at IS NOT NULL;
"""

run_sql(sql)


### Q48 — Average Resolution Time by Priority

Calculate average ticket resolution time in hours for each priority.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject

**Solution:**


In [ ]:
sql = """
SELECT priority,ROUND(AVG(EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0),2) avg_resolution_hours
FROM support_tickets WHERE resolved_at IS NOT NULL GROUP BY priority;
"""

run_sql(sql)


### Q49 — Customers Inactive for 60 Days

Relative to the maximum order_date, find customers whose latest order was more than 60 days ago, including customers with no orders.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.customer_id,c.first_name,c.last_name,MAX(o.order_date) latest_order
FROM customers c LEFT JOIN orders o USING(customer_id) GROUP BY c.customer_id,c.first_name,c.last_name
HAVING MAX(o.order_date) IS NULL OR MAX(o.order_date)<(SELECT MAX(order_date)-60 FROM orders);
"""

run_sql(sql)


### Q50 — Quarterly Revenue

Return year, quarter, order_count, and revenue for completed orders.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT EXTRACT(YEAR FROM order_date)::int AS year,EXTRACT(QUARTER FROM order_date)::int AS quarter,
COUNT(*) order_count,SUM(total_amount) revenue FROM orders WHERE status='Completed' GROUP BY 1,2 ORDER BY 1,2;
"""

run_sql(sql)


## 6. Deduplication


### Q51 — Duplicate Customer Emails

Identify normalized email addresses that appear more than once in customer_staging.  
**Tables:** \
**`customer_staging:`** staging_id, source_customer_id, first_name, last_name, email, city, updated_at

**Solution:**


In [ ]:
sql = """
SELECT LOWER(TRIM(email)) normalized_email,COUNT(*) copies FROM customer_staging GROUP BY 1 HAVING COUNT(*)>1;
"""

run_sql(sql)


### Q52 — Keep Latest Customer Record

Deduplicate customer_staging by normalized email, keeping the row with the latest updated_at.  
**Tables:** \
**`customer_staging:`** staging_id, source_customer_id, first_name, last_name, email, city, updated_at

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY LOWER(TRIM(email)) ORDER BY updated_at DESC,staging_id DESC) rn FROM customer_staging)
SELECT * FROM x WHERE rn=1;
"""

run_sql(sql)


### Q53 — Duplicate Events

Identify duplicate events based on user_id, event_type, and event_ts.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page

**Solution:**


In [ ]:
sql = """
SELECT user_id,event_type,event_ts,COUNT(*) copies FROM events GROUP BY user_id,event_type,event_ts HAVING COUNT(*)>1;
"""

run_sql(sql)


### Q54 — Remove Duplicate Events in Result

Return one logical event per user_id, event_type, event_ts using ROW_NUMBER(). Prefer the highest event_id.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY user_id,event_type,event_ts ORDER BY event_id DESC) rn FROM events)
SELECT * FROM x WHERE rn=1;
"""

run_sql(sql)


### Q55 — Duplicate Payments

Identify orders with more than one payment row.  
**Tables:** \
**`payments:`** payment_id, order_id, payment_date, amount, payment_method, payment_status

**Solution:**


In [ ]:
sql = """
SELECT order_id,COUNT(*) payment_rows FROM payments GROUP BY order_id HAVING COUNT(*)>1;
"""

run_sql(sql)


### Q56 — Latest Payment per Order

Return only the most recent payment record for each order.  
**Tables:** \
**`payments:`** payment_id, order_id, payment_date, amount, payment_method, payment_status

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY payment_date DESC,payment_id DESC) rn FROM payments)
SELECT * FROM x WHERE rn=1;
"""

run_sql(sql)


### Q57 — Duplicate Product SKUs

Find duplicated normalized SKUs in product_staging.  
**Tables:** \
**`product_staging:`** staging_id, sku, product_name, category, unit_price, updated_at

**Solution:**


In [ ]:
sql = """
SELECT UPPER(TRIM(sku)) normalized_sku,COUNT(*) copies FROM product_staging GROUP BY 1 HAVING COUNT(*)>1;
"""

run_sql(sql)


### Q58 — Canonical Product Staging

Return one row per normalized SKU, keeping the latest updated_at and using staging_id as a tie-breaker.  
**Tables:** \
**`product_staging:`** staging_id, sku, product_name, category, unit_price, updated_at

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY UPPER(TRIM(sku)) ORDER BY updated_at DESC,staging_id DESC) rn FROM product_staging)
SELECT * FROM x WHERE rn=1;
"""

run_sql(sql)


### Q59 — Duplicate Daily Logs

Identify repeated pipeline log records having the same pipeline_name, log_level, message, and logged_at.  
**Tables:** \
**`pipeline_logs:`** log_id, pipeline_name, log_level, message, logged_at

**Solution:**


In [ ]:
sql = """
SELECT pipeline_name,log_level,message,logged_at,COUNT(*) copies FROM pipeline_logs
GROUP BY pipeline_name,log_level,message,logged_at HAVING COUNT(*)>1;
"""

run_sql(sql)


### Q60 — Deduplicated Log Feed

Return a deduplicated pipeline log feed, keeping the highest log_id for each duplicate group.  
**Tables:** \
**`pipeline_logs:`** log_id, pipeline_name, log_level, message, logged_at

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY pipeline_name,log_level,message,logged_at ORDER BY log_id DESC) rn FROM pipeline_logs)
SELECT * FROM x WHERE rn=1;
"""

run_sql(sql)


## 7. Top-N Problems


### Q61 — Top 3 Customers by Revenue

Return the top 3 customers by total completed-order revenue.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.customer_id,c.first_name,c.last_name,SUM(o.total_amount) revenue
FROM customers c JOIN orders o USING(customer_id) WHERE o.status='Completed'
GROUP BY c.customer_id,c.first_name,c.last_name ORDER BY revenue DESC LIMIT 3;
"""

run_sql(sql)


### Q62 — Second-Highest Company Salary

Return all employees whose salary equals the second-highest distinct salary in the company.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
SELECT * FROM employees WHERE salary=(SELECT MAX(salary) FROM employees WHERE salary<(SELECT MAX(salary) FROM employees));
"""

run_sql(sql)


### Q63 — Top 3 Salaries per Department

Return the employees in the top 3 distinct salary levels within each department.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT e.*,DENSE_RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) rnk FROM employees e)
SELECT * FROM x WHERE rnk<=3 ORDER BY department_id,rnk;
"""

run_sql(sql)


### Q64 — Latest Transaction per Customer

Return each customer's latest transaction from transactions.  
**Tables:** \
**`transactions:`** transaction_id, customer_id, transaction_ts, transaction_type, amount, channel

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT t.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY transaction_ts DESC,transaction_id DESC) rn FROM transactions t)
SELECT * FROM x WHERE rn=1;
"""

run_sql(sql)


### Q65 — Top 2 Products per Category

Rank products by units sold within category and return the top 2 products per category.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price

**Solution:**


In [ ]:
sql = """
WITH s AS (SELECT p.product_id,p.product_name,p.category,COALESCE(SUM(oi.quantity),0) units
FROM products p LEFT JOIN order_items oi USING(product_id) GROUP BY p.product_id,p.product_name,p.category),
r AS (SELECT *,DENSE_RANK() OVER(PARTITION BY category ORDER BY units DESC) rnk FROM s)
SELECT * FROM r WHERE rnk<=2 ORDER BY category,rnk;
"""

run_sql(sql)


### Q66 — Highest-Value Order per Month

Return the highest-value order in each month. Return ties.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT o.*,DENSE_RANK() OVER(PARTITION BY DATE_TRUNC('month',order_date) ORDER BY total_amount DESC) rnk FROM orders o)
SELECT * FROM x WHERE rnk=1;
"""

run_sql(sql)


### Q67 — Top 3 Cities by Revenue

Return the top 3 customer cities by total order revenue.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.city,SUM(o.total_amount) revenue FROM customers c JOIN orders o USING(customer_id)
GROUP BY c.city ORDER BY revenue DESC LIMIT 3;
"""

run_sql(sql)


### Q68 — Most Active Users

Return the 5 users with the highest number of events.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page

**Solution:**


In [ ]:
sql = """
SELECT user_id,COUNT(*) event_count FROM events GROUP BY user_id ORDER BY event_count DESC LIMIT 5;
"""

run_sql(sql)


### Q69 — Third-Latest Order per Customer

Return the third-latest order for customers who have at least 3 orders.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT o.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date DESC,order_id DESC) rn FROM orders o)
SELECT * FROM x WHERE rn=3;
"""

run_sql(sql)


### Q70 — Top Employee by Project Load

Return the employee or employees with the greatest sum of allocation_pct across all project assignments.  
**Tables:** \
**`employees:`** employee_id, first_name, last_name, email, department_id, manager_id, hire_date, salary, job_title \
**`project_assignments:`** assignment_id, project_id, employee_id, role_on_project, allocation_pct

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT employee_id,SUM(allocation_pct) total_allocation FROM project_assignments GROUP BY employee_id),
r AS (SELECT *,DENSE_RANK() OVER(ORDER BY total_allocation DESC) rnk FROM x)
SELECT e.employee_id,e.first_name,e.last_name,r.total_allocation FROM r JOIN employees e USING(employee_id) WHERE r.rnk=1;
"""

run_sql(sql)


## 8. Real-World Data Problems


### Q71 — Customer 360 Snapshot

Build one result per customer with signup_date, segment, lifetime_orders, lifetime_revenue, latest_order_date, and open_ticket_count.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject

**Solution:**


In [ ]:
sql = """
WITH o AS (SELECT customer_id,COUNT(*) lifetime_orders,SUM(total_amount) lifetime_revenue,MAX(order_date) latest_order_date FROM orders GROUP BY customer_id),
t AS (SELECT customer_id,COUNT(*) FILTER (WHERE status='Open') open_ticket_count FROM support_tickets GROUP BY customer_id)
SELECT c.customer_id,c.signup_date,c.segment,COALESCE(o.lifetime_orders,0) lifetime_orders,COALESCE(o.lifetime_revenue,0) lifetime_revenue,
o.latest_order_date,COALESCE(t.open_ticket_count,0) open_ticket_count FROM customers c LEFT JOIN o USING(customer_id) LEFT JOIN t USING(customer_id);
"""

run_sql(sql)


### Q72 — Potentially Stuck Pipelines

Find pipeline runs/log groups with ERROR logs or with a duration greater than 90 minutes. Return pipeline_name, run_date, error_count, and duration_minutes.  
**Tables:** \
**`pipeline_logs:`** log_id, pipeline_name, log_level, message, logged_at

**Solution:**


In [ ]:
sql = """
WITH x AS (SELECT pipeline_name,logged_at::date AS run_date,COUNT(*) FILTER (WHERE log_level='ERROR') error_count,
EXTRACT(EPOCH FROM(MAX(logged_at)-MIN(logged_at)))/60 duration_minutes FROM pipeline_logs GROUP BY pipeline_name,logged_at::date)
SELECT * FROM x WHERE error_count>0 OR duration_minutes>90 ORDER BY run_date,pipeline_name;
"""

run_sql(sql)


### Q73 — Checkout Conversion Funnel

Using events, calculate distinct users who viewed a product, added to cart, and purchased, plus view-to-purchase conversion percentage.  
**Tables:** \
**`events:`** event_id, user_id, event_type, event_ts, session_id, page

**Solution:**


In [ ]:
sql = """
SELECT COUNT(DISTINCT user_id) FILTER (WHERE event_type='product_view') viewed_users,
COUNT(DISTINCT user_id) FILTER (WHERE event_type='add_to_cart') cart_users,
COUNT(DISTINCT user_id) FILTER (WHERE event_type='purchase') purchased_users,
ROUND(100.0*COUNT(DISTINCT user_id) FILTER (WHERE event_type='purchase')/
NULLIF(COUNT(DISTINCT user_id) FILTER (WHERE event_type='product_view'),0),2) view_to_purchase_pct FROM events;
"""

run_sql(sql)


### Q74 — Revenue Reconciliation

Compare each order's total_amount with the sum of its order_items and return orders where the absolute difference is greater than $0.01.  
**Tables:** \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price

**Solution:**


In [ ]:
sql = """
SELECT o.order_id,o.total_amount,SUM(oi.quantity*oi.unit_price) item_total,
o.total_amount-SUM(oi.quantity*oi.unit_price) difference
FROM orders o JOIN order_items oi USING(order_id) GROUP BY o.order_id,o.total_amount
HAVING ABS(o.total_amount-SUM(oi.quantity*oi.unit_price))>0.01;
"""

run_sql(sql)


### Q75 — Customer Churn Candidates

Relative to the dataset's maximum order_date, find customers with at least 2 historical orders but no order in the last 90 days.  
**Tables:** \
**`customers:`** customer_id, first_name, last_name, email, city, province, segment, signup_date \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
SELECT c.customer_id,c.first_name,c.last_name,COUNT(o.order_id) historical_orders,MAX(o.order_date) latest_order
FROM customers c JOIN orders o USING(customer_id) GROUP BY c.customer_id,c.first_name,c.last_name
HAVING COUNT(o.order_id)>=2 AND MAX(o.order_date)<(SELECT MAX(order_date)-90 FROM orders);
"""

run_sql(sql)


### Q76 — Payment Failure Rate

For each month, calculate total payment attempts, failed payments, and failure_rate_pct.  
**Tables:** \
**`payments:`** payment_id, order_id, payment_date, amount, payment_method, payment_status

**Solution:**


In [ ]:
sql = """
SELECT DATE_TRUNC('month',payment_date)::date AS month,COUNT(*) total_attempts,
COUNT(*) FILTER (WHERE payment_status='Failed') failed_payments,
ROUND(100.0*COUNT(*) FILTER (WHERE payment_status='Failed')/COUNT(*),2) failure_rate_pct
FROM payments GROUP BY 1 ORDER BY 1;
"""

run_sql(sql)


### Q77 — Support SLA Breaches

Assume SLA targets are High=24h, Medium=48h, Low=72h. Return resolved/closed tickets that exceeded their SLA.  
**Tables:** \
**`support_tickets:`** ticket_id, customer_id, assigned_employee_id, created_at, resolved_at, priority, status, subject

**Solution:**


In [ ]:
sql = """
SELECT ticket_id,priority,created_at,resolved_at,
ROUND(EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0,2) resolution_hours,
CASE priority WHEN 'High' THEN 24 WHEN 'Medium' THEN 48 ELSE 72 END sla_hours
FROM support_tickets WHERE resolved_at IS NOT NULL
AND EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0 >
CASE priority WHEN 'High' THEN 24 WHEN 'Medium' THEN 48 ELSE 72 END;
"""

run_sql(sql)


### Q78 — Pipeline Error Trend

For each pipeline and day, calculate total logs, error logs, and error_rate_pct.  
**Tables:** \
**`pipeline_logs:`** log_id, pipeline_name, log_level, message, logged_at

**Solution:**


In [ ]:
sql = """
SELECT pipeline_name,logged_at::date AS log_date,COUNT(*) total_logs,
COUNT(*) FILTER (WHERE log_level='ERROR') error_logs,
ROUND(100.0*COUNT(*) FILTER (WHERE log_level='ERROR')/COUNT(*),2) error_rate_pct
FROM pipeline_logs GROUP BY pipeline_name,logged_at::date ORDER BY log_date,pipeline_name;
"""

run_sql(sql)


### Q79 — Product Reorder Candidates

Return active products with stock_quantity below reorder_level, along with units sold in the last 30 days relative to the dataset's max order date.  
**Tables:** \
**`products:`** product_id, sku, product_name, category, unit_price, stock_quantity, reorder_level, active \
**`order_items:`** order_item_id, order_id, product_id, quantity, unit_price \
**`orders:`** order_id, customer_id, sales_employee_id, order_date, status, total_amount

**Solution:**


In [ ]:
sql = """
WITH mx AS (SELECT MAX(order_date) max_date FROM orders),
s AS (SELECT oi.product_id,SUM(oi.quantity) units_30d FROM order_items oi JOIN orders o USING(order_id),mx
WHERE o.order_date BETWEEN mx.max_date-29 AND mx.max_date GROUP BY oi.product_id)
SELECT p.product_id,p.product_name,p.stock_quantity,p.reorder_level,COALESCE(s.units_30d,0) units_sold_last_30d
FROM products p LEFT JOIN s USING(product_id) WHERE p.active AND p.stock_quantity<p.reorder_level;
"""

run_sql(sql)


### Q80 — Suspicious Transaction Pattern

Find customers who had at least 3 transactions within the same calendar day and whose daily transaction amount exceeded $1,000.  
**Tables:** \
**`transactions:`** transaction_id, customer_id, transaction_ts, transaction_type, amount, channel

**Solution:**


In [ ]:
sql = """
SELECT customer_id,transaction_ts::date AS transaction_date,COUNT(*) transaction_count,SUM(amount) daily_amount
FROM transactions GROUP BY customer_id,transaction_ts::date HAVING COUNT(*)>=3 AND SUM(amount)>1000 ORDER BY daily_amount DESC;
"""

run_sql(sql)
